In [12]:
"""
=============================================================================
 DATA CLEANING & LABELING  |  LLM Semantic Router — MLOps Pre-processing
=============================================================================

## Why Data Cleaning & Labeling Is the Critical First Gate in an MLOps Pipeline

In any production MLOps pipeline, raw data arriving from an external source
(an API, a data lake, a database replica) is **never** training-ready. It
contains noise, inconsistencies, label ambiguity, and rows that would silently
corrupt a classifier's decision boundary. Skipping this step doesn't just hurt
model accuracy — it invalidates your entire evaluation framework, because
metrics computed on mislabeled or malformed samples are themselves meaningless.

### This step satisfies two distinct engineering concerns:

**① Data Cleaning (defensive data contracts)**
Raw data must be validated against an explicit schema contract before it enters
the ML pipeline. This includes: enforcing non-null constraints on feature and
label columns, stripping whitespace artefacts, removing structurally ambiguous
rows (ties, unknowns), and persisting audit-level logs of every row dropped and
why. In production (e.g., an Apache Spark job feeding a feature store), this
layer is the boundary enforced by a Great Expectations suite or a dbt test.

**② Label Engineering (business logic as code)**
Labels are not a property of the raw data — they are a *decision* encoding
domain knowledge. Here, the routing tier assigned to each prompt is derived
from which model *won* the human preference judgment. This is a deliberate,
defensible labeling strategy: if a small local model (Tier 1) satisfies the
user, the prompt is simple; if only a large API model (Tier 2) wins, the prompt
is complex. Encoding this logic inside a versioned class (`ArenaDataLabeler`)
ensures it is auditable, reproducible, and testable — not buried in a notebook.

**The output `data/labeled_data.csv` is the sole input artifact for all
downstream pipeline stages (DatabaseManager, feature engineering, model
training). Its integrity is the foundation everything else rests on.**

=============================================================================
"""

# ── Standard Library ──────────────────────────────────────────────────────────
import logging
import os
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

# ── Third-Party ───────────────────────────────────────────────────────────────
import pandas as pd

# ── Logging (timestamped, named logger — mirrors Python logging best practice)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)s]  %(name)s — %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("SemanticRouter.ArenaDataLabeler")


# ══════════════════════════════════════════════════════════════════════════════
#  DATACLASS: LabelingConfig
#  ─────────────────────────────────────────────────────────────────────────────
#  Encapsulates all business-logic constants in one place so the labeling
#  strategy is: (a) easy to audit, (b) easy to extend, and (c) never
#  scattered as magic strings across the codebase.
#
#  In a production MLOps system this config would be loaded from a YAML file
#  (e.g. via Hydra or OmegaConf) so model lists can be updated without
#  touching source code — critical for A/B testing new routing tiers.
# ══════════════════════════════════════════════════════════════════════════════
@dataclass
class LabelingConfig:
    """
    Immutable configuration for the Arena labeling strategy.

    Attributes
    ----------
    tier1_models : frozenset
        Model identifiers whose wins indicate a Tier 1 (local/small) prompt.
    tier2_models : frozenset
        Model identifiers whose wins indicate a Tier 2 (API/large) prompt.
    label_tier1 : str
        Canonical string label written to the Target_Tier column.
    label_tier2 : str
        Canonical string label written to the Target_Tier column.
    winner_col : str
        Column in raw data recording which model won ('model_a' / 'model_b').
    model_a_col : str
        Column holding the name of model A in each arena matchup.
    model_b_col : str
        Column holding the name of model B in each arena matchup.
    prompt_col : str
        Column containing the raw user prompt text.
    valid_winner_values : frozenset
        Exact strings that count as a decisive (non-tie) outcome.
    """
    tier1_models: frozenset = field(default_factory=lambda: frozenset({
        "koala-13b",
        "vicuna-13b",
        "alpaca-13b",
        "chatglm-6b",
        "dolly-v2-12b",
        "stablelm-tuned-alpha-7b",
        "oasst-pythia-12b",
    }))

    tier2_models: frozenset = field(default_factory=lambda: frozenset({
        "gpt-4",
        "gpt-3.5-turbo",
        "claude-v1",
        "claude-instant-v1",
    }))

    label_tier1:          str = "Tier_1_Local"
    label_tier2:          str = "Tier_2_API"
    winner_col:           str = "winner"
    model_a_col:          str = "model_a"
    model_b_col:          str = "model_b"
    prompt_col:           str = "user_prompt"

    # Only these two values represent a decisive model win.
    # 'tie' and 'tie (bothbad)' are excluded — they carry no routing signal.
    valid_winner_values: frozenset = field(default_factory=lambda: frozenset({
        "model_a",
        "model_b",
    }))


# ══════════════════════════════════════════════════════════════════════════════
#  CLASS: ArenaDataLabeler
#  ─────────────────────────────────────────────────────────────────────────────
#  Responsibility: own the complete ETL flow from raw arena CSV to a clean,
#  labeled DataFrame ready for the DatabaseManager ingestion step.
#
#  Intentional design choices:
#  • Config is injected, not hardcoded → testable with mock configs.
#  • Each cleaning/labeling concern lives in its own private method →
#    easy to unit-test in isolation and easy to override in a subclass.
#  • A `run_pipeline()` method orchestrates the stages in an explicit order,
#    mirroring the run() method of an Apache Beam / Spark pipeline DAG.
# ══════════════════════════════════════════════════════════════════════════════
class ArenaDataLabeler:
    """
    Cleans the raw LMSYS Chatbot Arena CSV and applies model-winner–based
    routing tier labels to produce a training-ready labeled dataset.

    Parameters
    ----------
    raw_path    : path to `data/raw_arena_data.csv` (HuggingFace API output).
    output_path : path where `data/labeled_data.csv` will be written.
    config      : LabelingConfig instance (uses defaults if not supplied).
    """

    # Columns we actually need — reading only these saves memory on large CSVs
    _REQUIRED_COLUMNS = {"winner", "model_a", "model_b", "user_prompt"}

    def __init__(
        self,
        raw_path:    str | Path,
        output_path: str | Path,
        config:      Optional[LabelingConfig] = None,
    ):
        self.raw_path    = Path(raw_path)
        self.output_path = Path(output_path)
        self.config      = config or LabelingConfig()
        self._df: Optional[pd.DataFrame] = None   # internal working DataFrame

        logger.info(
            "ArenaDataLabeler initialised  |  raw='%s'  output='%s'",
            self.raw_path, self.output_path,
        )

    # ══════════════════════════════════════════════════════════════════════════
    #  STAGE 1 — Load
    # ══════════════════════════════════════════════════════════════════════════
    def _load_raw(self) -> pd.DataFrame:
        """
        Read the raw CSV, selecting only the columns needed for labeling.

        Raises
        ------
        FileNotFoundError  — raw CSV is missing (HuggingFace step not run).
        ValueError         — required columns are absent from the file.
        """
        if not self.raw_path.exists():
            raise FileNotFoundError(
                f"Raw data file not found at '{self.raw_path}'.\n"
                "  → Ensure the HuggingFace API download step has been run "
                "and the file is saved to data/raw_arena_data.csv before "
                "executing this labeling pipeline."
            )

        logger.info("Loading raw CSV  →  %s", self.raw_path)
        df = pd.read_csv(self.raw_path, usecols=list(self._REQUIRED_COLUMNS))

        # ── Column contract validation ────────────────────────────────────────
        missing_cols = self._REQUIRED_COLUMNS - set(df.columns)
        if missing_cols:
            raise ValueError(
                f"Raw CSV is missing expected columns: {missing_cols}.\n"
                f"  Found: {list(df.columns)}"
            )

        logger.info(
            "Raw data loaded  |  shape BEFORE cleaning = %s  |  columns = %s",
            df.shape, list(df.columns),
        )
        self._log_separator("RAW DATA — SHAPE & WINNER DISTRIBUTION")
        print(f"  Shape (rows × cols)  :  {df.shape[0]:,} × {df.shape[1]}")
        print(f"  Winner value counts  :\n{df['winner'].value_counts().to_string()}\n")

        return df

    # ══════════════════════════════════════════════════════════════════════════
    #  STAGE 2 — Clean
    # ══════════════════════════════════════════════════════════════════════════
    def _clean(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Apply all data-quality cleaning rules and log the impact of each step.

        Rules applied (in order):
          1. Drop rows where user_prompt is null.
          2. Drop rows where user_prompt is an empty / whitespace-only string.
          3. Drop rows where winner is not a decisive outcome (i.e., ties).
          4. Strip leading/trailing whitespace from text columns.

        Each rule is logged with the row delta so data quality audits can
        pinpoint exactly where volume was lost — a standard expectation in
        production data pipelines (cf. dbt source freshness / row count tests).
        """
        c   = self.config
        log = self._drop_log   # helper defined below

        # ── Rule 1: Null user_prompt ──────────────────────────────────────────
        before = len(df)
        df = df.dropna(subset=[c.prompt_col])
        log("Dropped — null user_prompt", before, len(df))

        # ── Rule 2: Empty / whitespace-only user_prompt ───────────────────────
        before = len(df)
        df = df[df[c.prompt_col].str.strip() != ""]
        log("Dropped — empty/whitespace user_prompt", before, len(df))

        # ── Rule 3: Non-decisive winner (ties and unknown values) ─────────────
        before = len(df)
        df = df[df[c.winner_col].isin(c.valid_winner_values)]
        log("Dropped — tie / unknown winner", before, len(df))

        # ── Rule 4: Strip whitespace from text fields (normalisation) ─────────
        df[c.prompt_col]   = df[c.prompt_col].str.strip()
        df[c.model_a_col]  = df[c.model_a_col].str.strip()
        df[c.model_b_col]  = df[c.model_b_col].str.strip()
        df[c.winner_col]   = df[c.winner_col].str.strip()

        logger.info("Cleaning complete  |  shape AFTER cleaning = %s", df.shape)
        self._log_separator("AFTER CLEANING — SHAPE")
        print(f"  Shape (rows × cols)  :  {df.shape[0]:,} × {df.shape[1]}\n")

        return df.reset_index(drop=True)

    # ══════════════════════════════════════════════════════════════════════════
    #  STAGE 3 — Label
    # ══════════════════════════════════════════════════════════════════════════
    def _apply_labels(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Derive `Target_Tier` from the winner column and the two tier lists.

        Algorithm
        ---------
        1. Resolve the winning model's name:
               winner == 'model_a'  →  winning_model = model_a column value
               winner == 'model_b'  →  winning_model = model_b column value

        2. Map the winning model name to a tier label:
               winning_model ∈ tier1_models  →  'Tier_1_Local'
               winning_model ∈ tier2_models  →  'Tier_2_API'
               otherwise                     →  NaN  (will be dropped)

        3. Drop rows where the winning model is not in either defined tier
           (e.g. 'llama-13b', 'fastchat-t5-3b' — unlabeled models).
           These rows carry ambiguous routing signal and must not bias training.
        """
        c = self.config

        # ── Step 1: Resolve winning model name ───────────────────────────────
        # Use numpy-style vectorised where() for speed on large DataFrames
        df["winning_model"] = df[c.winner_col].map({
            "model_a": df[c.model_a_col],   # Series — pandas aligns by index
            "model_b": df[c.model_b_col],
        })
        # The dict-map approach above doesn't work for Series values directly;
        # use numpy.where for correct vectorised resolution:
        import numpy as np
        df["winning_model"] = np.where(
            df[c.winner_col] == "model_a",
            df[c.model_a_col],
            df[c.model_b_col],
        )

        # ── Step 2: Map winning model → tier label ────────────────────────────
        def _assign_tier(model_name: str) -> Optional[str]:
            if model_name in c.tier1_models:
                return c.label_tier1
            if model_name in c.tier2_models:
                return c.label_tier2
            return None   # model outside both tier lists → will be dropped

        df["Target_Tier"] = df["winning_model"].apply(_assign_tier)

        # ── Step 3: Drop rows with no tier assignment ─────────────────────────
        before = len(df)
        df = df.dropna(subset=["Target_Tier"])
        dropped = before - len(df)
        logger.info(
            "Label assignment  |  dropped %d rows (winning model not in any tier).",
            dropped,
        )

        # ── Step 4: Log winning model coverage for audit purposes ─────────────
        self._log_separator("WINNING MODEL → TIER MAPPING (model coverage)")
        coverage = (
            df.groupby(["winning_model", "Target_Tier"])
              .size()
              .reset_index(name="count")
              .sort_values("count", ascending=False)
        )
        print(coverage.to_string(index=False))
        print()

        return df.reset_index(drop=True)

    # ══════════════════════════════════════════════════════════════════════════
    #  STAGE 4 — Finalise & Export
    # ══════════════════════════════════════════════════════════════════════════
    def _finalise(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Select only the columns needed by downstream pipeline stages,
        print the final distribution report, and write the labeled CSV.
        """
        c = self.config
        labeled_df = df[[c.prompt_col, "Target_Tier"]].copy()

        # ── Final distribution report ─────────────────────────────────────────
        counts   = labeled_df["Target_Tier"].value_counts()
        percents = labeled_df["Target_Tier"].value_counts(normalize=True) * 100

        self._log_separator("FINAL LABELED DATASET — DISTRIBUTION REPORT")
        print(f"  Total labeled rows   :  {len(labeled_df):,}")
        print(f"  Final shape          :  {labeled_df.shape}\n")
        print(f"  {'Target_Tier':<20} {'Count':>8}  {'%':>7}  Visual")
        print(f"  {'─'*20}  {'─'*8}  {'─'*7}  {'─'*40}")

        for tier in counts.index:
            bar     = "█" * int(percents[tier] / 2) + "░" * (50 - int(percents[tier] / 2))
            print(f"  {tier:<20}  {counts[tier]:>8,}  {percents[tier]:>6.1f}%  {bar}")

        # ── Class imbalance advisory ──────────────────────────────────────────
        ratio = counts.max() / counts.min()
        print(f"\n  Imbalance ratio (max/min)  :  {ratio:.2f}x")
        if ratio > 1.5:
            logger.warning(
                "Class imbalance detected (ratio=%.2fx > 1.5x). "
                "Consider class_weight='balanced' in your classifier, "
                "or apply SMOTE / upsampling before training.",
                ratio,
            )
        else:
            logger.info("Class balance is acceptable (ratio=%.2fx ≤ 1.5x).", ratio)

        # ── Write to disk ─────────────────────────────────────────────────────
        self.output_path.parent.mkdir(parents=True, exist_ok=True)
        labeled_df.to_csv(self.output_path, index=False)
        logger.info(
            "Labeled CSV exported  →  %s  (%d rows)",
            self.output_path, len(labeled_df),
        )
        print(f"\n  ✅  Exported  →  {self.output_path}  ({len(labeled_df):,} rows)\n")

        return labeled_df

    # ══════════════════════════════════════════════════════════════════════════
    #  PUBLIC API: run_pipeline()
    # ══════════════════════════════════════════════════════════════════════════
    def run_pipeline(self) -> pd.DataFrame:
        """
        Execute all four ETL stages in sequence with full error handling.

        Pipeline DAG
        ────────────
        _load_raw()  →  _clean()  →  _apply_labels()  →  _finalise()
             │               │               │                 │
          (Stage 1)      (Stage 2)       (Stage 3)         (Stage 4)
           Load           Clean           Label          Export CSV

        Returns
        -------
        pd.DataFrame
            Final labeled DataFrame with columns: ['user_prompt', 'Target_Tier'].
            Identical to the content written to `data/labeled_data.csv`.

        Raises
        ------
        FileNotFoundError  — raw CSV not found.
        ValueError         — schema or data contract violation.
        Exception          — any unexpected error (logged before re-raise).
        """
        self._log_separator("ArenaDataLabeler — PIPELINE START", char="═")
        labeled_df = pd.DataFrame()

        try:
            # ── Stage 1: Load ─────────────────────────────────────────────────
            raw_df = self._load_raw()

            # ── Stage 2: Clean ────────────────────────────────────────────────
            clean_df = self._clean(raw_df)

            # ── Stage 3: Label ────────────────────────────────────────────────
            labeled_raw_df = self._apply_labels(clean_df)

            # ── Stage 4: Finalise & Export ────────────────────────────────────
            labeled_df = self._finalise(labeled_raw_df)

        except FileNotFoundError as exc:
            logger.error(
                "Pipeline aborted — raw file not found.\n  Details: %s", exc
            )
            raise

        except ValueError as exc:
            logger.error(
                "Pipeline aborted — data contract violation.\n  Details: %s", exc
            )
            raise

        except Exception as exc:
            logger.error(
                "Pipeline aborted — unexpected error: %s", exc, exc_info=True
            )
            raise

        self._log_separator("ArenaDataLabeler — PIPELINE COMPLETE ✅", char="═")
        return labeled_df

    # ══════════════════════════════════════════════════════════════════════════
    #  PRIVATE HELPERS
    # ══════════════════════════════════════════════════════════════════════════
    @staticmethod
    def _log_separator(title: str = "", char: str = "─", width: int = 65) -> None:
        """Print a titled separator line for readable notebook output."""
        print(f"\n  {char * width}")
        if title:
            print(f"  {title}")
        print(f"  {char * width}")

    @staticmethod
    def _drop_log(reason: str, before: int, after: int) -> None:
        """Log a single data-cleaning step with row delta."""
        dropped = before - after
        pct     = (dropped / before * 100) if before else 0
        logger.info(
            "%-42s  dropped=%6d  (%5.2f%%)  remaining=%d",
            reason, dropped, pct, after,
        )


# ══════════════════════════════════════════════════════════════════════════════
#  NOTEBOOK EXECUTION
# ══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    import os

    notebook_dir = os.path.abspath('')
    raw_path    = os.path.join(notebook_dir, "notebooks-training", "data", "raw_arena_data.csv")
    output_path = os.path.join(notebook_dir, "notebooks-training", "data", "labeled_arena_data.csv")

    labeler    = ArenaDataLabeler(raw_path, output_path)
    labeled_df = labeler.run_pipeline()

    print(f"✅ Labeling complete! {len(labeled_df)} rows saved to:\n   {output_path}")
    labeled_df







2026-05-02 23:20:53  [INFO]  SemanticRouter.ArenaDataLabeler — ArenaDataLabeler initialised  |  raw='d:\ToanFIles\LLM-Semantic-Router\notebooks-training\data\raw_arena_data.csv'  output='d:\ToanFIles\LLM-Semantic-Router\notebooks-training\data\labeled_arena_data.csv'
2026-05-02 23:20:53  [INFO]  SemanticRouter.ArenaDataLabeler — Loading raw CSV  →  d:\ToanFIles\LLM-Semantic-Router\notebooks-training\data\raw_arena_data.csv



  ═════════════════════════════════════════════════════════════════
  ArenaDataLabeler — PIPELINE START
  ═════════════════════════════════════════════════════════════════


2026-05-02 23:20:54  [INFO]  SemanticRouter.ArenaDataLabeler — Raw data loaded  |  shape BEFORE cleaning = (33000, 4)  |  columns = ['model_a', 'model_b', 'winner', 'user_prompt']
2026-05-02 23:20:54  [INFO]  SemanticRouter.ArenaDataLabeler — Dropped — null user_prompt                  dropped=     0  ( 0.00%)  remaining=33000
2026-05-02 23:20:54  [INFO]  SemanticRouter.ArenaDataLabeler — Dropped — empty/whitespace user_prompt      dropped=     0  ( 0.00%)  remaining=33000
2026-05-02 23:20:54  [INFO]  SemanticRouter.ArenaDataLabeler — Dropped — tie / unknown winner              dropped=  9706  (29.41%)  remaining=23294
2026-05-02 23:20:54  [INFO]  SemanticRouter.ArenaDataLabeler — Cleaning complete  |  shape AFTER cleaning = (23294, 4)
2026-05-02 23:20:54  [INFO]  SemanticRouter.ArenaDataLabeler — Label assignment  |  dropped 5586 rows (winning model not in any tier).
2026-05-02 23:20:54  [INFO]  SemanticRouter.ArenaDataLabeler — Class balance is acceptable (ratio=1.10x ≤ 1.5x).
2026-0


  ─────────────────────────────────────────────────────────────────
  RAW DATA — SHAPE & WINNER DISTRIBUTION
  ─────────────────────────────────────────────────────────────────
  Shape (rows × cols)  :  33,000 × 4
  Winner value counts  :
winner
model_a          11744
model_b          11550
tie (bothbad)     6263
tie               3443


  ─────────────────────────────────────────────────────────────────
  AFTER CLEANING — SHAPE
  ─────────────────────────────────────────────────────────────────
  Shape (rows × cols)  :  23,294 × 4


  ─────────────────────────────────────────────────────────────────
  WINNING MODEL → TIER MAPPING (model coverage)
  ─────────────────────────────────────────────────────────────────
          winning_model  Target_Tier  count
                  gpt-4   Tier_2_API   2888
             vicuna-13b Tier_1_Local   2667
          gpt-3.5-turbo   Tier_2_API   2524
              claude-v1   Tier_2_API   2390
              koala-13b Tier_1_Local   1901
      claud

In [13]:
import sqlite3
import logging
import os
import pandas as pd
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)s]  %(name)s — %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("SemanticRouter.DatabaseManager")


class DatabaseManager:
    """
    Manages ingestion and SQL-based retrieval of labeled prompt data.

    Simulates a production MLOps PostgreSQL pipeline using SQLite.
    All public methods use try-except-finally to guarantee connection cleanup.

    Parameters
    ----------
    db_path : str | Path
        Absolute path to the SQLite database file.
    labeled_csv_path : str | Path
        Absolute path to the labeled CSV produced by ArenaDataLabeler.
    """

    TABLE_NAME = "historical_prompts"

    def __init__(self, db_path: str | Path, labeled_csv_path: str | Path):
        self.db_path = Path(db_path)
        self.labeled_csv_path = Path(labeled_csv_path)

        # Ensure the parent directory exists
        self.db_path.parent.mkdir(parents=True, exist_ok=True)

        logger.info(
            "DatabaseManager initialised  |  db='%s'  csv='%s'",
            self.db_path,
            self.labeled_csv_path,
        )

    # ------------------------------------------------------------------
    # STAGE 1 — Ingest
    # ------------------------------------------------------------------
    def ingest_from_csv(self, if_exists: str = "replace") -> int:
        """
        Read labeled CSV and bulk-load it into the historical_prompts table.

        Parameters
        ----------
        if_exists : str
            Passed to DataFrame.to_sql — 'replace' (default) or 'append'.

        Returns
        -------
        int
            Number of rows written.

        Raises
        ------
        FileNotFoundError
            If the labeled CSV does not exist.
        """
        if not self.labeled_csv_path.exists():
            raise FileNotFoundError(
                f"Labeled CSV not found at '{self.labeled_csv_path}'.\n"
                "  → Run ArenaDataLabeler.run_pipeline() first."
            )

        logger.info("Reading labeled CSV → %s", self.labeled_csv_path)
        df = pd.read_csv(self.labeled_csv_path)

        conn = None
        try:
            conn = sqlite3.connect(self.db_path)
            df.to_sql(
                name=self.TABLE_NAME,
                con=conn,
                if_exists=if_exists,
                index=True,
                index_label="id",
            )
            conn.commit()
            row_count = len(df)
            logger.info(
                "Ingestion complete  |  table='%s'  rows=%d",
                self.TABLE_NAME,
                row_count,
            )
            return row_count

        except Exception as exc:
            logger.error("Ingestion failed: %s", exc)
            raise

        finally:
            if conn:
                conn.close()
                logger.info("Connection closed after ingest.")

    # ------------------------------------------------------------------
    # STAGE 2 — Fetch (CLO 1 core requirement)
    # ------------------------------------------------------------------
    def fetch_prompts(self, tier: str | None = None) -> pd.DataFrame:
        """
        Fetch labeled prompts from the database using pure SQL.

        Demonstrates CLO 1 SQL fetching via pandas.read_sql_query with
        a parameterised WHERE clause.

        Parameters
        ----------
        tier : str | None
            If provided, filter to 'Tier_1_Local' or 'Tier_2_API'.
            If None, returns all non-null prompts.

        Returns
        -------
        pd.DataFrame
        """
        if tier is not None:
            query = """
                SELECT id, user_prompt, Target_Tier
                FROM   historical_prompts
                WHERE  user_prompt IS NOT NULL
                  AND  Target_Tier = :tier
                ORDER  BY id
            """
            params = {"tier": tier}
        else:
            query = """
                SELECT id, user_prompt, Target_Tier
                FROM   historical_prompts
                WHERE  user_prompt IS NOT NULL
                ORDER  BY id
            """
            params = {}

        conn = None
        try:
            conn = sqlite3.connect(self.db_path)
            df = pd.read_sql_query(query, conn, params=params)
            logger.info(
                "fetch_prompts() returned %d rows  (tier_filter=%s)",
                len(df),
                tier or "ALL",
            )
            return df

        except Exception as exc:
            logger.error("fetch_prompts() failed: %s", exc)
            raise

        finally:
            if conn:
                conn.close()
                logger.info("Connection closed after fetch.")

    # ------------------------------------------------------------------
    # STAGE 3 — Aggregate Summary
    # ------------------------------------------------------------------
    def fetch_summary(self) -> pd.DataFrame:
        """
        Return a GROUP BY aggregate confirming correct label distribution.

        SQL used:
            SELECT Target_Tier, COUNT(*) AS prompt_count
            FROM   historical_prompts
            GROUP  BY Target_Tier
            ORDER  BY prompt_count DESC

        Returns
        -------
        pd.DataFrame
            Columns: Target_Tier, prompt_count, pct
        """
        query = """
            SELECT Target_Tier,
                   COUNT(*)                        AS prompt_count,
                   ROUND(
                       COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2
                   )                               AS pct
            FROM   historical_prompts
            WHERE  user_prompt IS NOT NULL
            GROUP  BY Target_Tier
            ORDER  BY prompt_count DESC
        """

        conn = None
        try:
            conn = sqlite3.connect(self.db_path)
            df = pd.read_sql_query(query, conn)
            logger.info("fetch_summary() returned:\n%s", df.to_string(index=False))
            return df

        except Exception as exc:
            logger.error("fetch_summary() failed: %s", exc)
            raise

        finally:
            if conn:
                conn.close()
                logger.info("Connection closed after summary.")


# ──────────────────────────────────────────────────────────────────────────────
#  EXECUTION BLOCK
# ──────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":

    notebook_dir = os.path.abspath('')
    labeled_path = os.path.join(
        notebook_dir, "notebooks-training", "data", "labeled_arena_data.csv"
    )
    db_path = os.path.join(
        notebook_dir, "notebooks-training", "data", "router_logs.db"
    )

    db = DatabaseManager(db_path=db_path, labeled_csv_path=labeled_path)

    # Stage 1 — Ingest
    rows_written = db.ingest_from_csv(if_exists="replace")
    print(f"\n✅ Ingested {rows_written:,} rows into '{DatabaseManager.TABLE_NAME}'")

    # Stage 2 — Fetch (CLO 1 SQL requirement)
    sql_df = db.fetch_prompts()
    print(f"\n📋 Sample of fetched SQL data ({len(sql_df):,} rows total):")
    display(sql_df.head(10))

    # Stage 3 — Aggregate summary
    summary_df = db.fetch_summary()
    print("\n📊 Label distribution (SQL COUNT + GROUP BY):")
    display(summary_df)


2026-05-02 23:35:23  [INFO]  SemanticRouter.DatabaseManager — DatabaseManager initialised  |  db='d:\ToanFIles\LLM-Semantic-Router\notebooks-training\data\router_logs.db'  csv='d:\ToanFIles\LLM-Semantic-Router\notebooks-training\data\labeled_arena_data.csv'
2026-05-02 23:35:23  [INFO]  SemanticRouter.DatabaseManager — Reading labeled CSV → d:\ToanFIles\LLM-Semantic-Router\notebooks-training\data\labeled_arena_data.csv
2026-05-02 23:35:23  [INFO]  SemanticRouter.DatabaseManager — Ingestion complete  |  table='historical_prompts'  rows=17708
2026-05-02 23:35:23  [INFO]  SemanticRouter.DatabaseManager — Connection closed after ingest.
2026-05-02 23:35:23  [INFO]  SemanticRouter.DatabaseManager — fetch_prompts() returned 17708 rows  (tier_filter=ALL)
2026-05-02 23:35:23  [INFO]  SemanticRouter.DatabaseManager — Connection closed after fetch.



✅ Ingested 17,708 rows into 'historical_prompts'

📋 Sample of fetched SQL data (17,708 rows total):


,id,user_prompt,Target_Tier
0,0,What is the difference between OpenCL and CUDA?,Tier_1_Local
1,1,"Fuji vs. Nikon, which is better?",Tier_1_Local
2,2,How to build an arena for chatbots?,Tier_1_Local
3,3,When is it today?,Tier_1_Local
4,4,Count from 1 to 10 with step = 3,Tier_1_Local
5,5,"Emoji for ""sharing"". List 10",Tier_1_Local
6,6,How to parallelize a neural network?,Tier_1_Local
7,7,"A = 5, B =10, A+B=?",Tier_1_Local
8,8,What is the future of bitcoin?,Tier_1_Local
9,9,Make it more polite: I want to have dinner.,Tier_1_Local


2026-05-02 23:35:23  [INFO]  SemanticRouter.DatabaseManager — fetch_summary() returned:
 Target_Tier  prompt_count   pct
  Tier_2_API          9295 52.49
Tier_1_Local          8413 47.51
2026-05-02 23:35:23  [INFO]  SemanticRouter.DatabaseManager — Connection closed after summary.



📊 Label distribution (SQL COUNT + GROUP BY):


,Target_Tier,prompt_count,pct
0,Tier_2_API,9295,52.49
1,Tier_1_Local,8413,47.51


## ☁️ Data Source 3 — Cloud Data Retrieval (Hugging Face Hub)

### How this satisfies the "Cloud" requirement of CLO 1

The raw Arena data was streamed from the **Hugging Face Hub**, a managed cloud data
platform. This is architecturally equivalent to reading from AWS S3, Google Cloud
Storage, or Azure Data Lake.

```python
# Authenticated cloud fetch — physically reads from Hugging Face CDN:
dataset = load_dataset("lmsys/chatbot_arena_conversations")


In [15]:

"""
=============================================================================
 DATA SOURCE 4 — NoSQL Document Store Simulation
=============================================================================

Purpose
-------
Demonstrate "NoSQL" data retrieval as required by CLO 1.

Technique
---------
JSON files are the canonical storage format for Document Databases such as
MongoDB, CouchDB, and AWS DynamoDB. Each record in the JSON array is a
self-describing "document" — exactly how MongoDB stores BSON documents.

In production, the two functions below map directly to:
  - export_to_json()    →  db.collection.insertMany(documents)
  - query_by_tier()     →  db.collection.find({"Target_Tier": tier})
=============================================================================
"""

import json
import os
import logging
from pathlib import Path

logger = logging.getLogger("SemanticRouter.NoSQLStore")


class NoSQLDocumentStore:
    """
    Simulates a MongoDB-style Document Store using a local JSON file.

    In production, replace the json.dump / json.load calls with
    pymongo.collection.insert_many() / find() — the interface is identical.

    Parameters
    ----------
    json_path : str | Path
        Absolute path where the JSON document store will be written.
    """

    def __init__(self, json_path: str | Path):
        self.json_path = Path(json_path)
        self.json_path.parent.mkdir(parents=True, exist_ok=True)
        logger.info("NoSQLDocumentStore ready  |  store='%s'", self.json_path)

    # ------------------------------------------------------------------
    # WRITE — mirrors MongoDB insertMany()
    # ------------------------------------------------------------------
    def export_to_json(self, df, n_docs: int = 100) -> int:
        """
        Serialise the first `n_docs` rows of a DataFrame to JSON.

        Each row becomes an independent document (dict), matching the
        MongoDB document model exactly.

        Parameters
        ----------
        df : pd.DataFrame
            Source DataFrame (labeled_df from ArenaDataLabeler).
        n_docs : int
            Number of documents to export (default 100).

        Returns
        -------
        int  — number of documents written.
        """
        documents = (
            df.head(n_docs)
              .reset_index(drop=True)
              .rename_axis("doc_id")
              .reset_index()
              .to_dict(orient="records")
        )

        with open(self.json_path, "w", encoding="utf-8") as f:
            json.dump(documents, f, ensure_ascii=False, indent=2)

        logger.info(
            "export_to_json() wrote %d documents → %s", len(documents), self.json_path
        )
        return len(documents)

    # ------------------------------------------------------------------
    # READ + FILTER — mirrors MongoDB find({"Target_Tier": tier})
    # ------------------------------------------------------------------
    def query_by_tier(self, tier: str) -> list[dict]:
        """
        Load all documents from the JSON store and filter by Target_Tier.

        Uses list comprehension to replicate a MongoDB equality filter:
            db.prompts.find({"Target_Tier": tier})

        Parameters
        ----------
        tier : str
            Label to filter on — 'Tier_1_Local' or 'Tier_2_API'.

        Returns
        -------
        list[dict]  — matching documents.

        Raises
        ------
        FileNotFoundError — if the JSON store has not been created yet.
        """
        if not self.json_path.exists():
            raise FileNotFoundError(
                f"Document store not found at '{self.json_path}'.\n"
                "  → Call export_to_json() first."
            )

        with open(self.json_path, "r", encoding="utf-8") as f:
            all_docs = json.load(f)

        # List comprehension = NoSQL equality filter
        results = [doc for doc in all_docs if doc.get("Target_Tier") == tier]

        logger.info(
            "query_by_tier(tier='%s') → %d / %d documents matched",
            tier, len(results), len(all_docs),
        )
        return results


# ──────────────────────────────────────────────────────────────────────────────
#  EXECUTION BLOCK
# ──────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":

    notebook_dir   = os.path.abspath('')
    json_path      = os.path.join(
        notebook_dir, "notebooks-training", "data", "prompts_nosql.json"
    )
    labeled_csv    = os.path.join(
        notebook_dir, "notebooks-training", "data", "labeled_arena_data.csv"
    )

    import pandas as pd
    labeled_df = pd.read_csv(labeled_csv)

    store = NoSQLDocumentStore(json_path=json_path)

    # Write 100 documents to the JSON store
    n = store.export_to_json(labeled_df, n_docs=100)
    print(f"\n✅ Exported {n} documents to JSON document store:\n   {json_path}")

    # Query (NoSQL-style filter)
    tier2_docs = store.query_by_tier(tier="Tier_2_API")
    print(f"\n📄 NoSQL query result — Tier_2_API documents: {len(tier2_docs)}")
    for doc in tier2_docs[:3]:
        print(f"  [{doc['doc_id']}] {doc['user_prompt'][:80]}...")

    tier1_docs = store.query_by_tier(tier="Tier_1_Local")
    print(f"\n📄 NoSQL query result — Tier_1_Local documents: {len(tier1_docs)}")


2026-05-02 23:56:22  [INFO]  SemanticRouter.NoSQLStore — NoSQLDocumentStore ready  |  store='d:\ToanFIles\LLM-Semantic-Router\notebooks-training\data\prompts_nosql.json'
2026-05-02 23:56:22  [INFO]  SemanticRouter.NoSQLStore — export_to_json() wrote 100 documents → d:\ToanFIles\LLM-Semantic-Router\notebooks-training\data\prompts_nosql.json
2026-05-02 23:56:22  [INFO]  SemanticRouter.NoSQLStore — query_by_tier(tier='Tier_2_API') → 0 / 100 documents matched
2026-05-02 23:56:22  [INFO]  SemanticRouter.NoSQLStore — query_by_tier(tier='Tier_1_Local') → 100 / 100 documents matched



✅ Exported 100 documents to JSON document store:
   d:\ToanFIles\LLM-Semantic-Router\notebooks-training\data\prompts_nosql.json

📄 NoSQL query result — Tier_2_API documents: 0

📄 NoSQL query result — Tier_1_Local documents: 100


## ✅ CLO 1 Completion Summary — All Four Data Sources Covered

> **CLO 1:** *"Retrieve data from multiple data sources: SQL, NoSQL, APIs, Cloud."*

| Requirement | Implementation | Key Technique | File / Table |
|---|---|---|---|
| **API** | Hugging Face `load_dataset()` | Authenticated HTTPS REST call | `raw_arena_data.csv` |
| **Cloud** | Hugging Face Hub (CDN-distributed) | Token-authenticated cloud stream | `raw_arena_data.csv` |
| **SQL** | SQLite via `sqlite3` + `pandas.read_sql_query` | `SELECT … WHERE`, `COUNT … GROUP BY` | `router_logs.db → historical_prompts` |
| **NoSQL** | JSON Document Store via `json` module | Document serialisation + list-comprehension filter | `prompts_nosql.json` |

All four retrieval methods are production-portable:

- **API → Cloud:** Swap `load_dataset()` for `boto3.client('s3').get_object()`
- **SQL:** Swap `sqlite3.connect()` for `psycopg2.connect()` (PostgreSQL)
- **NoSQL:** Swap `json.dump/load` for `pymongo.collection.insert_many/find()` (MongoDB)
